# Train the Aircraft Vision model (free Colab)

**Always start with ▷ Run all** in the toolbar, not the ▶ button on a single box: the boxes depend on the ones above them.

Everything is saved to your Google Drive, so **if Colab stops (usage limit, disconnect, closed tab), nothing is lost.** Just come back later and do **Runtime → Run all** again. It skips what's done and carries on training from the last finished round.

**Step 1: download the photos (no GPU needed, so it doesn't use up your GPU time)**
- Runtime → Change runtime type → **CPU** → Save, then click **▷ Run all** in the toolbar (or Runtime → Run all).
- Allow it to connect to Google Drive when asked.
- It stops at the "Train" cell with a message telling you to switch to GPU. That's expected.

**Step 2: train (needs GPU)**
- Runtime → Change runtime type → **T4 GPU** → Save, then **Runtime → Run all**.
- If it stops because of the usage limit, wait until GPU is available again (often the next day) and **Run all** again.

At the end it downloads `aircraft_model.zip`. If any step fails, it stops with a message asking you to send the lines above it to Claude.

**Tip:** free Colab can disconnect if the tab sits untouched for a long time. Keep the tab open and check on it now and then. If it disconnects, just **Run all** again.

In [ ]:
import os, subprocess, json
os.environ['PYTHONUNBUFFERED'] = '1'   # show script output live

def run(cmd, must_succeed=True):
    """Run a command, show its output live, and stop the notebook with a clear message if it fails."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        msg = f'This step failed (exit code {p.returncode}). Copy the last lines above and send them to Claude.'
        if must_succeed:
            raise RuntimeError(msg)
        print('WARNING: ' + msg)

if not os.path.exists('/content/aircraft-vision'):
    run('git clone -q https://github.com/Themagicianleo/aircraft-vision.git /content/aircraft-vision')
else:
    run('git -C /content/aircraft-vision pull -q')
%cd /content/aircraft-vision/training
run('pip -q install timm onnx onnxruntime onnxscript')

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/aircraft-vision'   # everything that must survive is kept here
os.makedirs(f'{WORK}/data', exist_ok=True)
print('Saving to Google Drive:', WORK)

## Photos 1/2: FGVC-Aircraft (100 types). Done once, about 10 min

In [ ]:
if os.path.exists(f'{WORK}/data/fgvc_prepared/meta.json'):
    print('Already downloaded.')
else:
    import shutil
    # Runs as a separate program so it can't freeze the notebook.
    run('python -c "from av.data import prepare_fgvc; prepare_fgvc(\'/content/fgvc_raw\', workers=4)"')  # big raw download stays on this machine...
    print('Copying to Google Drive...')
    shutil.copytree('/content/fgvc_raw/fgvc_prepared', f'{WORK}/data/fgvc_prepared', dirs_exist_ok=True)  # ...only the shrunk copy (~1 GB) goes to Drive
    print('Done.')

## Photos 2/2: Wikimedia Commons (modern types + more airliner photos). Done once, can take an hour
If it's interrupted, Run all again and it continues. Types are listed in `training/commons_types.csv`.

In [ ]:
run(f'python fetch_commons.py --out {WORK}/data/commons --per-class 250', must_succeed=False)
n = sum(len(f) for _, _, f in os.walk(f'{WORK}/data/commons'))
print(f'Commons photos ready: {n}')

## Train (GPU). Picks up from the last finished round if it was interrupted

In [ ]:
import torch, shutil
if not torch.cuda.is_available():
    raise SystemExit('Photos are ready. Now switch to a GPU: Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all.')

EPOCHS = 40
hist = f'{WORK}/runs/effb0/history.json'
done = len(json.load(open(hist))) if os.path.exists(hist) else 0
print(f'Training rounds already finished: {done} of {EPOCHS}')

print('Copying photos from Drive to this machine (faster to read)...')
shutil.copytree(f'{WORK}/data', '/content/data_local', dirs_exist_ok=True)
run(f'python train.py --data-root /content/data_local --extra-data /content/data_local/commons '
    f'--out {WORK}/runs/effb0 --resume --workers 2 --epochs {EPOCHS} --img-size 320')

## Export to ONNX (fp32 + int8) and measure accuracy with the app's exact preprocessing

In [ ]:
run(f'python export.py --checkpoint {WORK}/runs/effb0/last.pt --out {WORK}/exported --int8 --calib-dir /content/data_local/fgvc_prepared/images')
run(f'python evaluate.py --onnx {WORK}/exported/aircraft.onnx --data-root /content/data_local --extra-data /content/data_local/commons')
run(f'python evaluate.py --onnx {WORK}/exported/aircraft_int8.onnx --labels {WORK}/exported/labels.json --data-root /content/data_local --extra-data /content/data_local/commons')

## Download
Pick the int8 model if its accuracy above is within ~1% of fp32 (it's 4× smaller). Otherwise keep fp32.

In [ ]:
USE_INT8 = False
import os, shutil
WORK = '/content/drive/MyDrive/aircraft-vision'
if not os.path.exists(f'{WORK}/exported/aircraft.onnx'):
    raise SystemExit('The model is not ready yet: training has not finished. Use Runtime → Run all (not just this box).')
pkg = '/content/aircraft_model'
os.makedirs(pkg, exist_ok=True)
shutil.copy(f'{WORK}/exported/' + ('aircraft_int8.onnx' if USE_INT8 else 'aircraft.onnx'), f'{pkg}/aircraft.onnx')
shutil.copy(f'{WORK}/exported/labels.json', f'{pkg}/labels.json')
if os.path.exists(f'{WORK}/data/commons/CREDITS.csv'):
    shutil.copy(f'{WORK}/data/commons/CREDITS.csv', f'{pkg}/PHOTO_CREDITS.csv')  # keep this: CC licences require crediting authors
shutil.make_archive('/content/aircraft_model', 'zip', pkg)
from google.colab import files
files.download('/content/aircraft_model.zip')